# Chat老师小课堂——DSA5102 Final Project
## 用强化学习训练 AI 玩贪吃蛇：完整实验框架（5 人组）

**定位：** 这是一份“项目蓝图 + 零基础教材 + 可运行代码骨架”，不是已经替你们得出结论的成品报告。最终报告必须填入你们真实运行得到的图、表、录像、失败案例和解释。

**正式任务：Option 2 — Reinforcement Learning**

课程要求的硬指标：

- 用强化学习训练机器玩经典游戏，并假设读者从未玩过；
- 可视化展示训练过程中的进步；
- 探索如何提高玩家强度并比较性能；
- 必须有 baseline、learning curves、多个 episodes 的评估；
- 提交一份可直接运行的 Jupyter notebook 和一个小组视频；
- 5 人视频总长 20 分钟，每人约 4 分钟，必须真人讲解；
- 长训练需附保存模型，并设置 TRAIN_MODEL=False；
- 报告中声明成员贡献和生成式 AI 的用途。

**截止日期：** 正式 PDF 只写 November 22nd，本框架不擅自添加年份。

## 使用路线（专业复核后的版本）

1. 全组先读 MDP、Q-learning 与实验设计；
2. Member 1 完成环境测试，并用 6/10/15 做规模 pilot；
3. Member 2 跑 Pure/Safe Random、Greedy 与 Tabular Q；
4. Member 3 跑 DQN、DDQN；
5. Member 4 在固定 DDQN 下比较 O1 11D 与 O2 30D；核心完成后再决定 Dueling 或 PPO；
6. Member 5 负责 reward ablation、多 seed 统计、录像和报告；
7. 最终每个人都要能解释全项目。

> 科研纪律：一次只改变一个主要因素；棋盘和表示是待检验设计，不是默认最优；测试集只做最终评估；没有真实结果前不写“显著提升”。


# 1. 从零理解机器学习与强化学习

## 1.1 三类学习问题

- **监督学习**：给输入和正确答案，如房价回归；模型学习输入→标签。
- **无监督学习**：只有输入，如聚类；寻找数据结构。
- **强化学习（RL）**：没有每一步的标准答案；智能体采取动作，从环境获得奖励，并要最大化长期总回报。

贪吃蛇没有“此刻唯一正确方向”的逐步标签。一个动作可能立即靠近食物，却在十步后把自己困死，所以需要学习长期后果。

## 1.2 MDP：把游戏写成数学问题

马尔可夫决策过程 MDP 为五元组

$$\mathcal M=(\mathcal S,\mathcal A,P,R,\gamma).$$

- $\mathcal S$：状态空间，例如蛇头、蛇身、方向、食物；
- $\mathcal A$：动作空间；
- $P(s'|s,a)$：执行动作后转移到新状态的概率；
- $R(s,a,s')$：即时奖励；
- $\gamma\in[0,1)$：折扣因子，控制未来奖励的重要程度。

从时刻 $t$ 起的回报：

$$G_t=r_{t+1}+\gamma r_{t+2}+\gamma^2r_{t+3}+\cdots.$$

$\gamma$ 越大越重视长期生存；过小会只顾眼前食物。这里建议从 0.95 或 0.99 开始，不要因为游戏“短”便武断设得很低。

## 1.3 Policy、value 与 Q-value

- **策略** $\pi(a|s)$：在状态 $s$ 选择动作 $a$ 的规则。
- **状态价值** $V^\pi(s)=\mathbb E_\pi[G_t|S_t=s]$：从该状态按策略继续，长期回报期望。
- **动作价值** $Q^\pi(s,a)=\mathbb E_\pi[G_t|S_t=s,A_t=a]$：先做动作 $a$ 再按策略继续的长期回报。

最优 Q 函数满足 Bellman optimality equation：

$$Q^*(s,a)=\mathbb E\left[r+\gamma\max_{a'}Q^*(s',a')\right].$$

含义：某动作的长期价值 = 眼前奖励 + 下一状态最好未来价值。DQN 的核心就是用神经网络逼近这个函数。

# 2. 游戏与实验总设计

## 2.1 规则（写给从未玩过的人）

在 $10\times10$ 网格上，蛇每一步必须向前移动一格。吃到食物后身体增长、分数加一，食物重新随机生成。撞墙或撞到身体则 episode 结束。目标不是“走得久”，而是尽可能多吃食物并避免把自己困住。

**核心固定设置：**

- 棋盘 10×10，初始长度 3；
- 每局设无进食步数上限，防止无限绕圈；
- 食物只生成在空格；
- 训练与评估使用不同随机种子；
- 所有算法使用同一环境版本和终止规则。

## 2.2 推荐动作：3 个相对动作

$$\mathcal A=\{\text{straight},\text{turn right},\text{turn left}\}.$$

相比“上下左右 + 屏蔽反向”，相对动作有三个优点：

1. 天然排除 180° 掉头；
2. 与危险信号 straight/right/left 完全对应；
3. 具有旋转对称性：朝上和朝右时“左转”含义相同。

若小组坚持 4 个绝对方向也可以，但必须正确 action masking。两种定义不可在不同算法间混用，否则比较不公平。

## 2.3 11 维观测：快，但并非完整 Markov state

$$o_t=[danger_{straight/right/left},direction_{4},foodRelative_{4}].$$

- 3 维危险：下一步是否撞墙/身体；
- 4 维当前朝向 one-hot；
- 4 维食物在蛇头左/右/上/下。

优点是低维、CPU 训练快、适合课程项目。局限是它没有完整蛇身形状：两个真实局面可有相同 11 维观测，却需要不同动作。因此严格说它是压缩 observation，可能产生 **state aliasing**，智能体面对部分可观测问题。

现实意义：现实决策也常只看到摘要特征。若模型到达性能天花板，原因可能不是算法差，而是信息不足。建议把“11 维表示限制”写入局限；有余力再做 full-grid/CNN 扩展，不作为核心必做。

## 2.4 奖励与游戏原生指标必须分开

核心奖励方案：

- R1 Sparse：吃食 +1，死亡 -1，其余 0；
- R2 Step penalty：R1 + 每步 -0.01；
- R3 Potential shaping：R1 加
  $$F(s,a,s')=\eta[\gamma\Phi(s')-\Phi(s)],$$
  其中 $\Phi(s)=-d_{Manhattan}(head,food)/D_{max}$。

Potential-based shaping 在标准条件下不改变最优策略，只让学习信号更密集；简单“靠近 +0.1、远离 -0.1”可能改变目标并诱发投机。

**跨奖励方案不可用 total reward 作为主比较指标**，因为奖励刻度本来不同。统一主指标必须是：

1. 每局吃到的食物数（score）；
2. 生存步数；
3. 死亡类型与绕圈率；
4. 达到某分数的比例。

现实意义：商业 KPI 也不能因内部奖励函数改了就宣称业务提升；必须回到外部、可解释的真实结果。

# 2A. 专业复核：原方案哪些是合理起点，哪些不能称为“最优”

## 结论先行

- **10×10 不是理论最优棋盘**：它只是可视化、难度和计算预算之间的工程折中，必须经规模预实验确认。
- **11 维不是最优状态维度**：它很省样本，却丢失蛇身全局结构；维度少不等于表示好。
- **DQN 合理**：动作离散、维度低、经验可重复利用，十分契合 Snake。
- **DDQN 合理且优先级高**：它只改变 target 计算，能形成干净的单变量实验。
- **Dueling 并非必然提升**：Snake 只有 3 个动作，动作数量少，论文所强调的“大量相近动作”优势可能较弱；它应由实验决定是否保留。
- **应加入 Tabular Q-learning**：11 维二值观测最多只有 $2^{11}=2048$ 个编码，Q 表只有 $2048\times3=6144$ 个值。它是从 Q-learning 到 DQN 最清楚的教学桥梁，也能检验神经网络是否真的必要。
- **PPO 可作为跨算法家族的备选挑战者**，但不应在 DQN 尚未跑稳时加入。

专业建议将主问题改为：

> 在计算预算受限的 Snake 中，性能瓶颈主要来自价值学习算法，还是来自状态表示的信息损失？

## 2A.1 为什么首先考虑 value-based 方法

Snake 的动作是 3 个离散选择。Value-based 方法直接估计每个动作的长期价值：

$$a_t=\arg\max_a Q(o_t,a).$$

它有三项适配性：

1. 离散动作少，逐动作输出 Q 很自然；
2. off-policy，可将旧经验放入 replay buffer 重复学习，样本效率较高；
3. 课程报告容易把 Bellman equation、target、loss 与改进机制讲清。

这不是说 value-based 永远最好，而是它在“低维 + 小离散动作 + 有限算力 + 初学团队”的约束下，具有较好的风险收益比。

## 2A.2 新的算法选择阶梯

### Level 0：非学习基线

Pure Random、Safe Random、Greedy；若有余力增加 Hamiltonian/A* 作为强规划基线。

### Level 1：Tabular Q-learning（新增，核心）

直接存 $Q(o,a)$ 表。它没有神经网络，能让全组先理解 TD error、探索和 bootstrap；也是判断 DQN 是否仅靠函数逼近增加复杂度的对照。

### Level 2：DQN（核心）

把 Q 表换成神经网络，观察泛化是否提升。

### Level 3：Double DQN（核心）

只更换 target 的选择/评价机制，检验过估计偏差。

### Level 4：Dueling DDQN（条件核心）

只有当时间足够且前三级稳定时加入。若结果无提升，也是一项可信发现。

### Level 5：PPO 或表示学习（只能二选一作为扩展）

- 想比较算法范式：选 PPO；
- 想研究 Snake 的关键瓶颈：选状态表示比较。**本项目更推荐后者。**

## 2A.3 算法取舍矩阵

| 方法 | 原理 | 适合点 | 不作为首选的原因 | 建议 |
|---|---|---|---|---|
| Tabular Q | 每个观测动作存一个值 | 11维二值态很小、最易理解 | 状态别名严重、难扩到图像 | 新增核心 RL baseline |
| SARSA | 用实际下一动作 bootstrap | on-policy，能研究探索风险 | 与 Q-learning 差异可能小 | 可替换 Tabular Q，不必都做 |
| DQN | 神经网络近似 Q | 离散动作、replay 高效 | 训练不稳、可能过估 | 核心 |
| Double DQN | 选择/评价分离 | 改动小、因果归因清楚 | 不保证得分总提高 | 核心 |
| Dueling | 分解 $V$ 与 $A$ | 多个动作价值接近时有效 | 本游戏仅 3 动作，收益不确定 | 条件核心 |
| PER/n-step | 优先抽样/多步回报 | 可提高样本效率 | 又增加多个变量与实现风险 | 后续可选，勿同时叠加 |
| Rainbow | 组合 6 类 DQN 改进 | 强且完整 | 无法知道哪个组件起作用，范围过大 | 不推荐初版 |
| REINFORCE | 直接做 policy gradient | 概念纯粹 | 方差高、样本利用率低 | 教学可讲，项目不优先 |
| A2C/PPO | actor-critic / clipped policy update | 跨范式、有成熟库 | on-policy 较耗样本，调参和解释更多 | 有余力选 PPO 一项 |
| SAC | 最大熵 actor-critic | 连续动作常很强 | 标准 SAC 面向连续控制，离散 Snake 过度复杂 | 不选 |
| MCTS/A* | 用已知模型搜索未来 | Snake 模型已知，可能很强 | 不是纯 RL | 强基线或 hybrid |
| 遗传/进化 | 直接优化策略参数 | 直观且无需梯度 | 样本效率低、偏离课程主线 | 不推荐 |

## 2A.3a 先用两条轴理解所有 RL 算法

### 是否学习环境模型

- **Model-free**：不显式学习 $P(s'|s,a)$，直接学价值或策略。Q-learning、DQN、PPO 属于此类。
- **Model-based**：已知或学习转移模型，再搜索/规划未来。Snake 规则已知，所以 A*、MCTS 很自然。

### 学什么对象

- **Value-based**：学 $Q(s,a)$，策略由 argmax 导出，如 Q-learning/DQN；
- **Policy-based**：直接学 $\pi_\theta(a|s)$，如 REINFORCE；
- **Actor-critic**：actor 学策略，critic 学价值并指导 actor，如 A2C/PPO。

这两个分类轴不能混为一谈。例如 PPO 是 model-free actor-critic；MCTS 是 model-based planning。

## 2A.3b Q-learning 与 SARSA：差别只在下一动作，却代表不同问题

Q-learning target：

$$r+\gamma\max_{a'}Q(s',a').$$

它假设下一步将采用贪心动作，即使当前行为策略还在 $\epsilon$-greedy 探索，因此是 **off-policy**。

SARSA target：

$$r+\gamma Q(s',a'),\qquad a'\sim\pi(\cdot|s').$$

它评估的是实际含探索的行为策略，因此是 **on-policy**。在“探索动作本身会造成危险”的任务中，SARSA 可能学得更保守。

为何不把二者都列为核心：11D Snake 上差异可能有趣，但实验预算有限；Tabular Q 已足以连接基础与 DQN。若小组研究“探索风险”，可用 SARSA 替换 Dueling，而不是继续增加方法。

## 2A.3c REINFORCE：直接让高回报动作更可能发生

策略网络输出动作概率 $\pi_\theta(a|s)$。Policy gradient 的核心形式：

$$\nabla_\theta J(\theta)=\mathbb E\left[G_t\nabla_\theta\log\pi_\theta(A_t|S_t)\right].$$

若一次轨迹回报高，就增加其中动作的 log probability；回报低则反向调整。

优点：直接优化随机策略，不需要对所有动作估 Q。缺点：完整轨迹回报噪声大、方差高、经验难以像 DQN replay 那样反复离策略使用。在只有 3 个动作的低维 Snake 上，它没有明显胜过 DQN 的先验理由，因此不优先。

## 2A.3d Actor-Critic 与 PPO

Actor 产生策略，critic 估计价值。常用 advantage：

$$A_t\approx \hat G_t-V_\phi(s_t),$$

表示动作结果比该状态通常水平好多少。PPO 用新旧策略概率比

$$r_t(\theta)=\frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{old}}(a_t|s_t)}$$

以及 clipped objective：

$$L^{CLIP}=\mathbb E[\min(r_tA_t,\operatorname{clip}(r_t,1-\epsilon,1+\epsilon)A_t)].$$

若新策略一步改得太多，ratio 被限制，减少破坏性更新。

为何 PPO 是合理备选：算法范式不同、有成熟实现、能直接输出随机策略。为何不默认选：它是 on-policy，旧数据很快过期，通常需要更多交互；还要理解 actor、critic、advantage、rollout、entropy 和 clip，给初学团队增加明显负担。

## 2A.3e Planning、Hybrid 与“最强玩家”不等于“最好 RL 实验”

Snake 的规则已知，可从当前局面模拟未来：

- A* 找蛇头到食物的短路；
- Hamiltonian cycle 保证按固定环移动，安全但可能效率低；
- MCTS 多次模拟未来动作并选择统计上更好者；
- Hybrid 可让 RL 决定目标或风险偏好，让搜索负责安全路径。

这些方法可能比低维 DQN 更强，但它们不是纯粹“从奖励学习”的主方案。最佳用法是：

1. 作为强非 RL baseline，显示 RL 距离规划上限多远；
2. 作为失败分析：RL 为什么会进死胡同而搜索不会；
3. 有创意时做 safety shield，但必须保留明确 RL 主体。

不要用强规划器取代 RL 后仍声称完成 Option 2。

## 2A.3f 一个选择模型的通用决策规则

按顺序问：

1. 动作离散还是连续？Snake 是小离散动作，DQN 合适。
2. 环境交互贵吗？若贵，off-policy replay 更有价值。
3. 状态是否完整？若 observation 丢信息，换更深网络也无法恢复。
4. 主要研究问题是什么？过估计→DDQN；探索风险→SARSA；策略更新稳定→PPO；全局规划→MCTS/表示。
5. 是否能做公平多 seed 比较？不能就减模型。
6. 团队能否口头推导并解释？不能理解的复杂模型不应成为主方法。

模型选择应由问题机制驱动，而不是排行榜或流行度。

## 2A.4 为何不是“模型越多越好”

若一次比较 DQN、DDQN、Dueling、PER、n-step、Rainbow、PPO，却每种只跑 1 个 seed，报告看似丰富，科学证据反而薄弱。

项目评分强调 experimental rigor。固定总预算下，优先级应为：

$$\text{正确环境}>\text{强基线}>\text{多 seed}>\text{清楚消融}>\text{更多模型名词}.$$

Rainbow 的原论文之所以有价值，不只因为组合组件，也因为做了详细 ablation。对课程项目，直接复制完整 Rainbow 会让实现与归因超出合理范围。

# 2B. 棋盘大小：10×10 是待验证的工程选择

棋盘边长 $B$ 增大时，格数是 $B^2$；但蛇身是有顺序的，长度 $L$ 的可能排列数量粗略达到

$$P(B^2,L)=\frac{(B^2)!}{(B^2-L)!},$$

还未计食物位置。难度不是随边长线性增长，而是组合爆炸。

不同尺寸的偏差：

- 6×6：训练快、便于 debug，但规划空间小，策略容易饱和；
- 10×10：通常能同时看到追食、避障和长身体规划，画面也清楚；
- 15×15：早期更容易存活，但找到食物更稀疏，episode 更长，训练成本显著增加；
- 20×20：对本项目可能把算力花在等待和探索，而非科学对比。

因此 10×10 是**候选主环境**，不是先验最优。

In [1]:
# 棋盘组合规模的数量级示意：固定蛇长 L=10
import math
def log10_permutations(n,k):
    return sum(math.log10(x) for x in range(n-k+1,n+1))
for B in [6,8,10,15,20]:
    print(f"{B}x{B}: log10 P(B^2,10) ≈ {log10_permutations(B*B,10):.1f}")

6x6: log10 P(B^2,10) ≈ 15.0
8x8: log10 P(B^2,10) ≈ 17.7
10x10: log10 P(B^2,10) ≈ 19.8
15x15: log10 P(B^2,10) ≈ 23.4
20x20: log10 P(B^2,10) ≈ 26.0


## 2B.1 正确的棋盘规模预实验

不要先训练所有算法。只用 Pure Random、Safe Random、Greedy 和短程 Tabular Q/DQN，在 $B\in\{6,10,15\}$ 上做 pilot：

记录：

- 每秒 environment steps；
- 随机/Greedy 的 score 与 episode length；
- 食物首次命中的稀疏程度；
- 50k steps 后是否出现学习信号；
- 一次完整多 seed 实验的预计时间；
- 游戏画面的可解释性。

**选 10×10 的可辩护标准：** 比 6×6 不易过早饱和；比 15×15 在固定预算内更稳定地产生学习；全套实验能在截止前至少跑 3–5 seeds。

若数据不支持，应改尺寸，而不是维护预设。

# 2C. 状态表示：信息含量比“维度最小”更重要

## O1：11D local features

危险 3 + 方向 4 + 食物相对方向 4。优点是极快、可解释；缺点是看不到身体全局布局，是部分可观测。

## O2：30D ray features（推荐扩展）

沿 8 个方向观察：

- 食物是否在该射线上；
- 到最近身体的逆距离；
- 到墙的逆距离；

共 $8\times3=24$ 维，再加方向 one-hot 4 维、归一化食物位移 $(\Delta x,\Delta y)$ 2 维，共 30 维。

它仍非严格 Markov，但能看到更远的墙、身体与通道，成本远低于网格 CNN。

## O3：Full structured grid（高配扩展）

三个 $B\times B$ channel：head、food、body-age，再加方向 4 维。在 10×10 时为 $3\times100+4=304$ 个数。body-age 记录从尾到头的顺序，仅用 body occupancy 不足以预测尾部下一步如何移动。

O3 信息最完整，但样本、网络和调试成本最高。

In [2]:
def mlp_params(input_dim,hidden=128,actions=3):
    return input_dim*hidden+hidden + hidden*hidden+hidden + hidden*actions+actions
for d,name in [(11,"O1 local"),(30,"O2 ray"),(304,"O3 grid-flat")]:
    print(name,"input dim",d,"MLP params",mlp_params(d))

target=mlp_params(30,128)
for d,name in [(11,"O1"),(30,"O2"),(304,"O3")]:
    h=min(range(16,257),key=lambda x:abs(mlp_params(d,x)-target))
    print(name,"若匹配约",target,"参数，hidden≈",h,"实际",mlp_params(d,h))

O1 local input dim 11 MLP params 18435
O2 ray input dim 30 MLP params 20867
O3 grid-flat input dim 304 MLP params 55939
O1 若匹配约 20867 参数，hidden≈ 137 实际 20964
O2 若匹配约 20867 参数，hidden≈ 128 实际 20867
O3 若匹配约 20867 参数，hidden≈ 57 实际 20865


## 2C.1 “最优维度”为什么不是一个独立问题

输入维度本身不能决定好坏：

- 11D 可因信息缺失产生 irreducible error；
- 304D 可能信息足够，却更难训练；
- 同一输入维度，不同编码可有完全不同的归纳偏置；
- 更高维网络通常参数更多，直接比较会混入模型容量差异。

状态表示实验应至少控制训练预算，并报告参数量。若要较严格归因，可：

1. O1 与 O2 都使用 MLP；
2. 调 hidden size 使参数量大致相当；
3. 固定 DDQN、reward 与 seeds；
4. O3 先用参数匹配的 flatten-MLP；CNN 作为另一项“表示 + 架构”联合实验，明确不能单独归因于表示。

## 2C.2 推荐的表示实验假设

- H-O1：O1 在早期样本效率更高，因为维度低且提供人工先验；
- H-O2：O2 在蛇较长时最终 score 更高，因为能看到远处身体和空间；
- H-O3：O3 可能有最高上限，但在当前训练预算内未必胜出；
- H-Alias：若 O1 在两种需要不同最优动作的真实局面给出同一编码，可直接证明 state aliasing。

最后一个假设可做成很漂亮的可视化：展示两张局面图、相同 11D 向量、不同安全长期动作。它比单纯报网络分数更能体现理解。

# 2D. 三种可完成作业的完整研究路线

## 路线 A：算法机制（最稳）

Random/Greedy → Tabular Q → DQN → DDQN → 可选 Dueling；固定 11D、10×10，再做 R1/R3 奖励比较。

优点：理论故事连续，最适合初学者。缺点：状态瓶颈未解决。

## 路线 B：状态表示（专业上最推荐）

Random/Greedy → DQN/DDQN；固定最佳算法，对比 O1 11D、O2 30D、可选 O3。再用少量奖励实验。

优点：紧扣 Snake 的真正困难，容易得到有意义的正/负结果。缺点：实现多种 observation，需仔细控制参数量。

## 路线 C：算法范式（有风险）

DQN/DDQN 与 PPO 比较，固定状态与奖励。

优点：value-based vs policy-based，方法跨度大。缺点：PPO on-policy 样本开销更高，公平预算定义困难，全组理论负担较大。

**建议选择 B 为主叙事、A 为学习阶梯：** Tabular Q→DQN→DDQN 建立算法基础，然后把主要创新放在 O1 vs O2。Dueling 和 PPO 只在核心实验全部完成后追加。

# 2E. 重新设计后的最小核心矩阵

为防组合爆炸，不做 algorithm × observation × reward 全因子。采用分阶段筛选：

1. **规模 pilot**：6/10/15，仅基线 + 短训练，选择主棋盘；
2. **学习阶梯**：O1 + Sparse 下比较 Tabular Q、DQN、DDQN；
3. **表示实验**：固定 DDQN + Sparse，比较 O1 与 O2；
4. **奖励实验**：固定上一步最佳表示与 DDQN，比较 Sparse 与 Potential shaping；
5. **条件扩展**：Dueling 或 PPO 二选一；
6. **最终冻结评估**：所有保留方案跑 5 seeds × 100 test episodes。

这使每一步的选择都由前一步证据支撑，也让五人分工能够汇合成一条因果链。

## 2F. 原始论文依据与阅读顺序

建议每位成员至少读摘要、方法公式和实验结论：

1. Watkins & Dayan (1992), Q-learning  
   https://www.gatsby.ucl.ac.uk/~dayan/papers/cjch.pdf
2. Mnih et al. (2015), Human-level control through deep reinforcement learning  
   https://doi.org/10.1038/nature14236
3. van Hasselt et al. (2016), Deep Reinforcement Learning with Double Q-learning  
   https://doi.org/10.1609/aaai.v30i1.10295
4. Wang et al. (2016), Dueling Network Architectures  
   https://proceedings.mlr.press/v48/wangf16.html
5. Ng, Harada & Russell (1999), Policy invariance under reward transformations  
   https://ai.stanford.edu/~ang/papers/shaping-icml99.pdf
6. Schulman et al. (2017), Proximal Policy Optimization Algorithms  
   https://arxiv.org/abs/1707.06347
7. Hessel et al. (2018), Rainbow  
   https://doi.org/10.1609/aaai.v32i1.11796
8. Henderson et al. (2018), Deep Reinforcement Learning that Matters  
   https://ojs.aaai.org/index.php/AAAI/article/view/11694
9. Agarwal et al. (2021), Statistical Precipice  
   https://papers.nips.cc/paper/2021/hash/f514cec81cb148559cf475e7426eed5e-Abstract.html

这些论文支持“为什么选、为何不选、怎样严谨比较”；它们不能替代你们在 Snake 上的真实实验。

### 2A–2F 学习检查

1. DQN 适合 Snake 的三个任务结构原因是什么？
2. 为什么 DDQN 比 Dueling 更适合做第一个改进实验？
3. Dueling 在 3 动作环境中为何可能收益有限？
4. 11D Q 表需要多少个 Q 值？神经网络仍可能有何价值？
5. 10×10 应通过哪些数据而不是直觉确定？
6. O2 比 O1 多了什么信息？仍缺什么？
7. 为什么不能直接比较 11D MLP 与 304D CNN 后断言“状态维度导致提升”？
8. 若总预算固定，你会选择 8 个算法×1 seed，还是 4 个算法×5 seeds？说明理由。

# 3. 专业复核后的研究问题与实验

| 阶段 | 实验 | 唯一主要变化 | 研究问题 |
|---|---|---|---|
| P0 | 环境验证 | 不训练 | 规则是否正确、可复现？ |
| P1 | 棋盘 pilot | 6/10/15 | 哪个尺寸在难度与预算间合适？ |
| E1 | 非学习基线 | 策略 | 学习方法是否超过简单规则？ |
| E2 | 学习阶梯 | Tabular Q/DQN/DDQN | 网络与 Double target 分别贡献什么？ |
| E3 | 表示实验 | O1 11D/O2 30D | 性能是否受状态信息限制？ |
| E4 | 奖励消融 | Sparse/Potential | 密集反馈能否加速且不投机？ |
| E5 | 条件扩展 | Dueling 或 PPO | 核心完成后是否值得扩展？ |
| E6 | 最终评估 | 冻结模型 | 哪个方案最终最强且稳定？ |
| E7 | 进步可视化 | checkpoint | 行为怎样随训练改变？ |

推荐主叙事：**从表格 Q 到深度 Q，再证明 Snake 的性能瓶颈可能来自 observation，而不只是网络结构。**


## 3.1 统一评估协议

推荐 **5 个训练种子**；算力不足最低 3 个，但必须坦白不确定性。每个种子独立初始化网络、探索和环境。每个训练种子的最终模型在固定的 100 个评估种子上、$\epsilon=0$ 运行。

实验单位是“训练种子”，不是单个 episode。100 个 episode 描述一个模型的表现分布；不能把它伪装成 100 次独立训练。

报告：

- seed-level mean ± standard deviation；
- 中位数与四分位数；
- 95% bootstrap CI（建议对训练种子的汇总值做）；
- 学习曲线横轴用 environment steps；
- AUC 或达到目标分数所需 steps 衡量样本效率；
- 测试集只在方案冻结后使用。

不要只展示最好种子或最好一局。

## 3.2 公平比较的控制变量

保持不变：环境版本、状态、动作、训练步数、网络宽度、optimizer、batch、replay 容量、评估种子、终止规则。

每次只改一个主要组件：

- E3：只把 DQN target 改为 DDQN target；
- E4：只换 dueling head；
- E5：固定最佳算法，只换 reward。

若同时换学习率、网络和奖励，结果无法归因。超参数应在 validation seeds 上选择，最终 test seeds 不参与选择。

# 4. 可复现配置

下面配置是起点，不是假装已经最优。先用短训练 smoke test，再冻结核心配置跑完整实验。

In [3]:
from dataclasses import dataclass, asdict
from pathlib import Path
import json, random, math, time
import numpy as np

PROJECT_DIR = Path.cwd()
TRAIN_MODEL = False  # 最终提交必须默认为 False

@dataclass
class Config:
    grid_size: int = 10
    max_steps_without_food: int = 100
    gamma: float = 0.99
    learning_rate: float = 1e-3
    batch_size: int = 128
    replay_capacity: int = 50_000
    learning_starts: int = 2_000
    target_update_steps: int = 1_000
    total_env_steps: int = 200_000
    epsilon_start: float = 1.0
    epsilon_end: float = 0.05
    epsilon_decay_steps: int = 100_000
    train_every: int = 4
    hidden_size: int = 128
    train_seeds: tuple = (11, 22, 33, 44, 55)
    eval_episodes: int = 100

CFG = Config()
print(json.dumps(asdict(CFG), indent=2))

{
  "grid_size": 10,
  "max_steps_without_food": 100,
  "gamma": 0.99,
  "learning_rate": 0.001,
  "batch_size": 128,
  "replay_capacity": 50000,
  "learning_starts": 2000,
  "target_update_steps": 1000,
  "total_env_steps": 200000,
  "epsilon_start": 1.0,
  "epsilon_end": 0.05,
  "epsilon_decay_steps": 100000,
  "train_every": 4,
  "hidden_size": 128,
  "train_seeds": [
    11,
    22,
    33,
    44,
    55
  ],
  "eval_episodes": 100
}


## 4.1 环境依赖

最终建议使用 NumPy、pandas、Matplotlib、seaborn、PyTorch。当前 notebook 的环境与 Random/Greedy 部分只需 NumPy；DQN 训练在没有 PyTorch 时会安全跳过。

项目锁定版本后，把环境写入 requirements.txt。不要在最终报告中静默安装依赖；应给出明确安装说明和版本。

In [4]:
import importlib.util
required = ["numpy","pandas","matplotlib","seaborn","torch"]
availability = {p: importlib.util.find_spec(p) is not None for p in required}
print(availability)
print("若需训练 DQN，请先在项目环境安装缺失库；TRAIN_MODEL=False 时报告仍应可运行。")

{'numpy': True, 'pandas': True, 'matplotlib': False, 'seaborn': False, 'torch': False}
若需训练 DQN，请先在项目环境安装缺失库；TRAIN_MODEL=False 时报告仍应可运行。


# 5. E0：Snake 环境与单元测试

## 理论意义

环境定义了 MDP。环境有 bug 时，后面所有“算法提升”都没有意义。最危险的错误包括：

- 食物生成在蛇身上；
- 掉头规则不一致；
- 蛇尾本步会移走，却被误判为碰撞；
- 不同算法使用不同终止条件；
- reset 后随机数不可复现。

## 现实意义

在真实 AI 系统中，数据管道和评价环境错误常比模型错误更致命。E0 是工程质量，也是科学有效性的前提。

In [5]:
class SnakeEnv:
    """纯 NumPy/标准库环境。坐标为 (x, y)，方向 0上1右2下3左；动作 0直行1右转2左转。"""
    DIRS = [(0,-1),(1,0),(0,1),(-1,0)]

    def __init__(self, size=10, reward_mode="sparse",
                 max_steps_without_food=100, gamma=0.99, shaping_eta=0.2):
        self.size=size
        self.reward_mode=reward_mode
        self.max_steps_without_food=max_steps_without_food
        self.gamma=gamma
        self.shaping_eta=shaping_eta
        self.rng=np.random.default_rng(0)
        self.reset()

    def reset(self, seed=None):
        if seed is not None:
            self.rng=np.random.default_rng(seed)
        c=self.size//2
        self.direction=1
        self.snake=[(c,c),(c-1,c),(c-2,c)]
        self.score=0
        self.steps=0
        self.steps_since_food=0
        self.done=False
        self.food=self._spawn_food()
        return self.observation()

    def _spawn_food(self):
        empty=[(x,y) for y in range(self.size) for x in range(self.size)
               if (x,y) not in self.snake]
        return empty[int(self.rng.integers(len(empty)))] if empty else None

    def _next(self, action):
        nd=(self.direction + (1 if action==1 else -1 if action==2 else 0))%4
        dx,dy=self.DIRS[nd]
        h=self.snake[0]
        return nd,(h[0]+dx,h[1]+dy)

    def _would_collide(self, action):
        _,nh=self._next(action)
        wall=not (0<=nh[0]<self.size and 0<=nh[1]<self.size)
        ate=(nh==self.food)
        # 未吃食时尾巴本步移走，因此进入旧尾格不算碰撞
        occupied=self.snake if ate else self.snake[:-1]
        return wall or nh in occupied

    def _phi(self):
        if self.food is None:
            return 0.0
        h=self.snake[0]
        d=abs(h[0]-self.food[0])+abs(h[1]-self.food[1])
        return -d/(2*(self.size-1))

    def observation(self):
        danger=[float(self._would_collide(a)) for a in (0,1,2)]
        direction=[float(self.direction==i) for i in range(4)]
        h=self.snake[0]; fx,fy=self.food
        food_rel=[float(fx<h[0]),float(fx>h[0]),float(fy<h[1]),float(fy>h[1])]
        return np.asarray(danger+direction+food_rel,dtype=np.float32)

    def step(self, action):
        if self.done:
            raise RuntimeError("episode 已结束，请先 reset")
        if action not in (0,1,2):
            raise ValueError("动作必须为 0直行/1右转/2左转")
        phi_old=self._phi()
        nd,nh=self._next(action)
        self.steps+=1; self.steps_since_food+=1
        wall=not (0<=nh[0]<self.size and 0<=nh[1]<self.size)
        ate=(nh==self.food)
        occupied=self.snake if ate else self.snake[:-1]
        body=nh in occupied
        timeout=self.steps_since_food>=self.max_steps_without_food

        if wall or body or timeout:
            self.done=True
            reason="wall" if wall else "self" if body else "timeout"
            reward=-1.0
            if self.reward_mode=="potential":
                reward+=self.shaping_eta*(self.gamma*0.0-phi_old)
            obs=self.observation()
        else:
            self.direction=nd
            self.snake.insert(0,nh)
            reward=0.0
            reason=None
            if ate:
                self.score+=1; self.steps_since_food=0
                self.food=self._spawn_food()
                reward=1.0
                if self.food is None:
                    self.done=True; reason="win"
            else:
                self.snake.pop()
            if self.reward_mode=="step_penalty" and not ate:
                reward-=0.01
            if self.reward_mode=="potential":
                phi_new=0.0 if self.done else self._phi()
                reward+=self.shaping_eta*(self.gamma*phi_new-phi_old)
            obs=self.observation()
        return obs,reward,self.done,{"score":self.score,"reason":reason,"steps":self.steps}

    def render_ascii(self):
        board=[["." for _ in range(self.size)] for _ in range(self.size)]
        if self.food is not None:
            board[self.food[1]][self.food[0]]="F"
        for i,(x,y) in enumerate(reversed(self.snake)):
            board[y][x]="o"
        hx,hy=self.snake[0]; board[hy][hx]="H"
        return "\n".join(" ".join(row) for row in board)

In [6]:
# E0 单元测试
e1=SnakeEnv(size=10); o1=e1.reset(seed=123)
e2=SnakeEnv(size=10); o2=e2.reset(seed=123)
assert o1.shape==(11,)
assert e1.food not in e1.snake
assert e1.food==e2.food
assert np.array_equal(o1,o2)

# 向前走一步，长度不变
old_len=len(e1.snake)
_,_,done,info=e1.step(0)
assert len(e1.snake)==old_len and not done

# 强制食物在下一格，吃后长度+1、score+1
e=SnakeEnv(size=10); e.reset(seed=1)
_,nh=e._next(0); e.food=nh
before=len(e.snake)
_,r,done,info=e.step(0)
assert len(e.snake)==before+1 and info["score"]==1 and r>0

print("E0 基础测试全部通过")
print(e.render_ascii())

E0 基础测试全部通过
. . . . . . . . . .
. . . . . . . . . .
. . . . . . . . . .
. . . . . . . . . .
. . . . . . . . . F
. . . o o o H . . .
. . . . . . . . . .
. . . . . . . . . .
. . . . . . . . . .
. . . . . . . . . .


## E0 需要保存的证据

- 初始局面和一次吃食后的截图；
- 撞墙、撞身体、超时终止的测试；
- 同一 seed 的食物序列可复现；
- observation 每一维的人工核对表；
- 进入旧尾格的边界测试。

**通过标准：** 所有测试通过后冻结环境版本，并记录环境代码哈希。之后不得为了让某算法得分更高而偷偷改环境。

# 6. E1：Random 与 Greedy 双基线

## M0 Random

在 3 个相对动作中均匀随机。它定义任务下界，证明游戏不是随便行动也能高分。

## M1 Greedy heuristic

从不立即碰撞的动作中，选使新蛇头到食物曼哈顿距离最小者。它是强非学习基线，回答：“是否真的需要 RL，还是简单规则已足够？”

Greedy 的现实局限：它只优化眼前距离，可能进入蛇身围成的口袋。RL 的价值不是自动保证更聪明，而是有机会从长期回报学到延迟后果。

In [7]:
def pure_random_policy(obs, env, rng):
    return int(rng.integers(3))

def safe_random_policy(obs, env, rng):
    safe=[a for a in range(3) if not env._would_collide(a)]
    return int(rng.choice(safe if safe else [0,1,2]))

def greedy_policy(obs, env, rng):
    candidates=[]
    for a in range(3):
        if env._would_collide(a):
            continue
        _,nh=env._next(a)
        d=abs(nh[0]-env.food[0])+abs(nh[1]-env.food[1])
        candidates.append((d,a))
    if not candidates:
        return 0
    best=min(d for d,a in candidates)
    tied=[a for d,a in candidates if d==best]
    return int(rng.choice(tied))

def run_episode(policy, seed, reward_mode="sparse", render=False):
    env=SnakeEnv(CFG.grid_size,reward_mode,CFG.max_steps_without_food,CFG.gamma)
    obs=env.reset(seed=seed)
    rng=np.random.default_rng(seed+10_000)
    total_reward=0.0
    while True:
        a=policy(obs,env,rng)
        obs,r,done,info=env.step(a)
        total_reward+=r
        if done:
            break
    if render:
        print(env.render_ascii())
    return {"seed":seed,"score":info["score"],"steps":info["steps"],
            "return":total_reward,"death":info["reason"]}

def evaluate_policy(policy, seeds):
    return [run_episode(policy,s) for s in seeds]

smoke_seeds=range(20)
for name,policy in [("PureRandom",pure_random_policy),("SafeRandom",safe_random_policy),("Greedy",greedy_policy)]:
    rows=evaluate_policy(policy,smoke_seeds)
    print(name,"mean score=",np.mean([x["score"] for x in rows]),
          "mean steps=",np.mean([x["steps"] for x in rows]))

PureRandom mean score= 0.05 mean steps= 16.75
SafeRandom mean score= 1.0 mean steps= 145.6
Greedy mean score= 19.5 mean steps= 156.45


## E1 假设、指标与解释

- H1a：Greedy 的平均 score 高于 Random；
- H1b：Greedy 仍存在可重复展示的局部贪心失败案例。

报告均值、标准差、中位数、箱线图、死亡类型。至少找一个 Greedy 因追食物进入死局的可视化案例。

**不要犯的错：** 若 Random 策略先过滤危险动作，它已不是完全随机下界，而是 safe-random。应明确命名。当前代码使用 safe-random；若要纯随机，应另跑一条 baseline。

# 6A. 棋盘规模 Pilot：先测再定 10×10

以下只运行非学习基线，目的是估计 episode 长度、得分尺度和计算速度，不用于决定最终“最好算法”。正式 pilot 还应给每个尺寸加入同预算的短 Tabular Q/DQN。

In [8]:
def run_episode_on_size(policy, seed, size, reward_mode="sparse"):
    env=SnakeEnv(size=size,reward_mode=reward_mode,
                 max_steps_without_food=max(50,size*10),gamma=CFG.gamma)
    obs=env.reset(seed=seed)
    rng=np.random.default_rng(seed+10_000)
    total=0.0
    while True:
        action=policy(obs,env,rng)
        obs,r,done,info=env.step(action)
        total+=r
        if done: break
    return {"size":size,"seed":seed,"score":info["score"],
            "steps":info["steps"],"return":total,"death":info["reason"]}

PILOT_ROWS=[]
for B in [6,10,15]:
    for name,policy in [("PureRandom",pure_random_policy),
                        ("SafeRandom",safe_random_policy),
                        ("Greedy",greedy_policy)]:
        start=time.perf_counter()
        rs=[run_episode_on_size(policy,s,B) for s in range(100)]
        elapsed=time.perf_counter()-start
        PILOT_ROWS.append({
            "size":B,"policy":name,
            "mean_score":float(np.mean([r["score"] for r in rs])),
            "mean_steps":float(np.mean([r["steps"] for r in rs])),
            "env_steps_per_sec":float(sum(r["steps"] for r in rs)/max(elapsed,1e-9))
        })
for row in PILOT_ROWS: print(row)

{'size': 6, 'policy': 'PureRandom', 'mean_score': 0.12, 'mean_steps': 7.13, 'env_steps_per_sec': 29978.136538349343}
{'size': 6, 'policy': 'SafeRandom', 'mean_score': 2.88, 'mean_steps': 93.88, 'env_steps_per_sec': 30890.85636503112}
{'size': 6, 'policy': 'Greedy', 'mean_score': 12.57, 'mean_steps': 61.0, 'env_steps_per_sec': 22868.07668565097}
{'size': 10, 'policy': 'PureRandom', 'mean_score': 0.11, 'mean_steps': 17.17, 'env_steps_per_sec': 34480.75045948417}
{'size': 10, 'policy': 'SafeRandom', 'mean_score': 1.63, 'mean_steps': 158.45, 'env_steps_per_sec': 26508.517858592226}
{'size': 10, 'policy': 'Greedy', 'mean_score': 18.43, 'mean_steps': 144.93, 'env_steps_per_sec': 19622.7365421179}
{'size': 15, 'policy': 'PureRandom', 'mean_score': 0.15, 'mean_steps': 39.57, 'env_steps_per_sec': 51351.86342628697}
{'size': 15, 'policy': 'SafeRandom', 'mean_score': 0.55, 'mean_steps': 191.45, 'env_steps_per_sec': 33863.43299323989}
{'size': 15, 'policy': 'Greedy', 'mean_score': 25.29, 'mean_ste

## 怎样解释 Pilot

- 如果 6×6 Greedy 很快接近高分，可能区分度不足；
- 如果 15×15 的 episode 很长、短训练几乎吃不到食物，完整实验成本过高；
- 10×10 若兼具学习信号、方法差异与可承受耗时，才正式冻结；
- 不同尺寸的 raw score 不完全可比，因为最大可吃食物数不同；跨尺寸可辅助报告 normalized score，例如 score/$B^2$，但主实验应固定尺寸。

当前输出只是 baseline smoke test，不足以单独确认 10×10。

# 6B. 三种 observation 的代码接口

所有 agent 只依赖 observation function。这样可以固定环境动力学，只替换信息表示。

In [9]:
RAY_DIRS=[(0,-1),(1,-1),(1,0),(1,1),(0,1),(-1,1),(-1,0),(-1,-1)]

def observation_11(env):
    return env.observation()

def observation_rays_30(env):
    features=[]
    hx,hy=env.snake[0]
    body=set(env.snake[1:])
    for dx,dy in RAY_DIRS:
        x,y=hx,hy; distance=0
        food_inv=0.0; body_inv=0.0
        while True:
            x+=dx; y+=dy; distance+=1
            if not (0<=x<env.size and 0<=y<env.size):
                wall_inv=1.0/distance
                break
            if (x,y)==env.food and food_inv==0:
                food_inv=1.0/distance
            if (x,y) in body and body_inv==0:
                body_inv=1.0/distance
        features.extend([food_inv,body_inv,wall_inv])
    direction=[float(env.direction==i) for i in range(4)]
    dx=(env.food[0]-hx)/(env.size-1)
    dy=(env.food[1]-hy)/(env.size-1)
    return np.asarray(features+direction+[dx,dy],dtype=np.float32)

def observation_grid_304(env):
    B=env.size
    head=np.zeros((B,B),dtype=np.float32)
    food=np.zeros((B,B),dtype=np.float32)
    body_age=np.zeros((B,B),dtype=np.float32)
    hx,hy=env.snake[0]; head[hy,hx]=1.0
    if env.food is not None:
        fx,fy=env.food; food[fy,fx]=1.0
    L=max(len(env.snake)-1,1)
    for idx,(x,y) in enumerate(env.snake[1:],start=1):
        body_age[y,x]=(len(env.snake)-idx)/L
    direction=np.asarray([float(env.direction==i) for i in range(4)],dtype=np.float32)
    return np.concatenate([head.ravel(),food.ravel(),body_age.ravel(),direction])

test_env=SnakeEnv(10); test_env.reset(seed=7)
print("O1",observation_11(test_env).shape)
print("O2",observation_rays_30(test_env).shape)
print("O3",observation_grid_304(test_env).shape)

O1 (11,)
O2 (30,)
O3 (304,)


## Observation 实现注意

当前 SnakeEnv 内部训练函数默认调用 11D observation。正式表示实验应让环境 step 返回完整内部状态，或由 wrapper 在每步调用指定 observation function；不能训练时用 O2、评估时误用 O1。

O3 在不同棋盘尺寸的维度不同；若要跨尺寸泛化，更适合 CNN + padding 或 fully convolutional network，而不是固定 flatten MLP。

# 6C. Tabular Q-learning：理解深度强化学习前的桥梁

对 11 个二值特征，先把向量编码成 $0$–$2047$ 的整数：

$$index(o)=\sum_{i=0}^{10}2^i o_i.$$

Q 表形状为 $2048\times3$。更新仍是

$$Q[o,a]\leftarrow Q[o,a]+\alpha(r+\gamma\max_{a'}Q[o',a']-Q[o,a]).$$

它和 DQN 的目标相同；区别只是 DQN 用共享参数在不同观测间泛化。若 Tabular Q 与 DQN 接近，说明神经网络未必带来价值；若都受限，则问题可能是 11D state aliasing。

In [10]:
def obs_to_index(obs):
    bits=(np.asarray(obs)>0.5).astype(int)
    return int(sum(int(v)<<i for i,v in enumerate(bits)))

def train_tabular_q(seed=11,total_steps=30_000,alpha=0.1,gamma=0.99):
    rng=np.random.default_rng(seed)
    env=SnakeEnv(10,"sparse",CFG.max_steps_without_food,gamma)
    obs=env.reset(seed=seed)
    Q=np.zeros((2**11,3),dtype=np.float32)
    episode_scores=[]

    for step in range(total_steps):
        frac=min(step/max(total_steps*.6,1),1.0)
        eps=1.0+frac*(0.05-1.0)
        s=obs_to_index(obs)
        action=int(rng.integers(3)) if rng.random()<eps else int(Q[s].argmax())
        next_obs,reward,done,info=env.step(action)
        ns=obs_to_index(next_obs)
        target=reward if done else reward+gamma*float(Q[ns].max())
        Q[s,action]+=alpha*(target-Q[s,action])
        obs=next_obs
        if done:
            episode_scores.append(info["score"])
            obs=env.reset(seed=int(rng.integers(1_000_000_000)))
    return Q,episode_scores

def qtable_policy(Q):
    def policy(obs,env,rng):
        return int(Q[obs_to_index(obs)].argmax())
    return policy

Q_SMOKE,TABULAR_TRAIN_SCORES=train_tabular_q(total_steps=20_000)
tab_rows=evaluate_policy(qtable_policy(Q_SMOKE),range(500,600))
print("Tabular Q smoke test（非正式结果）:",summarize_rows(tab_rows)
      if "summarize_rows" in globals() else
      {"mean_score":np.mean([r["score"] for r in tab_rows])})

Tabular Q smoke test（非正式结果）: {'mean_score': np.float64(12.82)}


## Tabular Q 的实验意义与局限

**理论意义：** 将 Bellman update 与神经网络解耦；全组先看懂 Q 表如何因经验改变。

**现实意义：** 复杂模型不一定必要。若简单模型在紧凑特征上已经足够，部署成本更低、可解释性更高。

**局限：**

- Q-learning 的经典收敛结论要求马尔可夫状态、充分探索和合适学习率条件；
- 11D 是部分观测，理论条件不完全满足；
- 未访问或稀少访问的编码学不到；
- 表格无法在相似但不相同的 observation 之间平滑泛化。

正式实验需与 DQN 使用相同 environment steps、train seeds 和 test protocol。

### 6A–6C 学习检查

1. 为什么只跑非学习 baseline 还不能最终确定棋盘大小？
2. O2 的 inverse distance 为什么可能比 raw distance 更容易缩放？
3. O3 为什么需要 body-age，而不仅是 body occupancy？
4. Tabular Q 有 6144 个值，为何仍可能输给 DQN？
5. Tabular Q 与 DQN 都表现差时，应优先怀疑算法还是 observation？如何设计实验区分？

### E0–E1 学习检查

1. 为什么环境测试属于科学方法，而不仅是“程序员工作”？
2. 11 维观测为什么可能不满足马尔可夫性？
3. Random 与 safe-random 哪个更强？报告中能否混称？
4. Greedy 得分高于 DQN 时，是否说明 RL 理论错误？
5. 设计一个贪婪追食物会失败的蛇身形状。

# 7. E2：Vanilla DQN

## 7.1 从 Q-learning 到 DQN

表格型 Q-learning 更新：

$$Q(s,a)\leftarrow Q(s,a)+\alpha\left[r+\gamma\max_{a'}Q(s',a')-Q(s,a)\right].$$

方括号是 TD error：新证据给出的目标与旧估计之差。状态太多时无法存 Q 表，DQN 用神经网络 $Q_\theta(o,a)$ 输入 11 维观测、输出 3 个动作值。

目标值：

$$y=r+\gamma(1-done)\max_{a'}Q_{\theta^-}(o',a').$$

损失：

$$L(\theta)=\mathbb E[\operatorname{Huber}(Q_\theta(o,a)-y)].$$

done 时未来价值为零，避免把下一局价值接到本局死亡后。

## 7.2 DQN 三个稳定组件

1. **Experience replay**：保存转移 $(o,a,r,o',done)$，随机采样，减少连续样本相关性并重复利用经验。
2. **Target network**：用较慢更新的 $\theta^-$ 计算目标，避免网络一边追目标、一边目标也剧烈移动。
3. **$\epsilon$-greedy**：以概率 $\epsilon$ 随机探索，否则选最大 Q；$\epsilon$ 随训练下降。

现实类比：回放像从案例库随机抽题复习；target network 像一段时间内固定评分标准；探索像给新方案少量试错预算。

常见失败：忘记 target detach、评估时仍有探索、不同算法 epsilon 日程不一致、把 terminal next-Q 加入目标。

In [11]:
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    import torch.optim as optim
    TORCH_AVAILABLE=True
except ImportError:
    TORCH_AVAILABLE=False
    print("PyTorch 尚未安装：理论、环境和基线仍可运行；DQN 训练单元将跳过。")

if TORCH_AVAILABLE:
    class QNetwork(nn.Module):
        def __init__(self, obs_dim=11, n_actions=3, hidden=128):
            super().__init__()
            self.net=nn.Sequential(
                nn.Linear(obs_dim,hidden),nn.ReLU(),
                nn.Linear(hidden,hidden),nn.ReLU(),
                nn.Linear(hidden,n_actions))
        def forward(self,x):
            return self.net(x)

    class DuelingQNetwork(nn.Module):
        def __init__(self, obs_dim=11, n_actions=3, hidden=128):
            super().__init__()
            self.trunk=nn.Sequential(nn.Linear(obs_dim,hidden),nn.ReLU(),
                                     nn.Linear(hidden,hidden),nn.ReLU())
            self.value=nn.Linear(hidden,1)
            self.advantage=nn.Linear(hidden,n_actions)
        def forward(self,x):
            h=self.trunk(x)
            v=self.value(h); a=self.advantage(h)
            return v+a-a.mean(dim=1,keepdim=True)

    print("PyTorch models ready")

PyTorch 尚未安装：理论、环境和基线仍可运行；DQN 训练单元将跳过。


## 7.3 为什么使用 Huber loss 与梯度裁剪

平方误差对巨大 TD error 的梯度也巨大，训练早期可能被少量异常转移主导。Huber loss 在误差小时像平方误差、误差大时近似绝对误差：

$$
\ell_\delta(e)=
\begin{cases}
\frac12e^2,&|e|\le\delta,\\
\delta(|e|-\frac12\delta),&|e|>\delta.
\end{cases}
$$

再用 gradient norm clipping 防止一次更新过大。它们提升数值稳定性，但不能修复错误状态、奖励或 target。

# 8. E3：Double DQN

Vanilla DQN 的 max 同时“选择”与“评价”动作。带噪估计中取最大值会偏向被偶然高估的动作，产生 overestimation bias。

Double DQN 分离两步：

$$a^*=\arg\max_a Q_\theta(o',a),$$

$$y_{DDQN}=r+\gamma(1-done)Q_{\theta^-}(o',a^*).$$

在线网络负责选择，目标网络负责评价。它不是两套完全独立智能体，也不保证任何数据集上得分都更高。

**实验假设：**

- DDQN 的估计 Q 与实际 Monte Carlo return 间正偏差更小；
- 多训练种子的学习曲线波动较低；
- 最终 score 是否提高由数据回答。

现实意义：预测系统常在多个带噪候选中选最大者，赢家诅咒会导致乐观偏差；分开选择与评估是普适思想。

# 9. E4：Dueling DDQN

Dueling architecture 将输出拆成：

- $V(o)$：当前局面整体有多好；
- $A(o,a)$：动作 $a$ 相对平均动作有多好。

为解决 $V$ 与 $A$ 可任意平移、不可辨识的问题，组合为

$$Q(o,a)=V(o)+A(o,a)-\frac1{|\mathcal A|}\sum_{a'}A(o,a').$$

在许多动作效果相近的安全空地，先学习状态价值可能更高效；临近危险时 advantage 再区分动作。

**公平对比：** DDQN 与 Dueling-DDQN 使用相同 target 公式、训练预算与 hidden size；同时报告参数量，避免把“更多参数”误当结构优势。

主要指标：学习曲线 AUC、达到 score 阈值所需步数、最终 score、多种子方差。

In [12]:
from collections import deque, namedtuple
Transition=namedtuple("Transition","obs action reward next_obs done")

class ReplayBuffer:
    def __init__(self,capacity):
        self.data=deque(maxlen=capacity)
    def add(self,*args):
        self.data.append(Transition(*args))
    def sample(self,batch_size,rng):
        idx=rng.choice(len(self.data),size=batch_size,replace=False)
        return [self.data[int(i)] for i in idx]
    def __len__(self):
        return len(self.data)

def epsilon_at(step,cfg=CFG):
    frac=min(step/cfg.epsilon_decay_steps,1.0)
    return cfg.epsilon_start+frac*(cfg.epsilon_end-cfg.epsilon_start)

print("epsilon:",[(s,round(epsilon_at(s),3)) for s in [0,50_000,100_000,200_000]])

epsilon: [(0, 1.0), (50000, 0.525), (100000, 0.05), (200000, 0.05)]


In [13]:
if TORCH_AVAILABLE:
    def compute_td_loss(batch,online,target,optimizer,gamma,double_dqn):
        obs=torch.tensor(np.stack([b.obs for b in batch]),dtype=torch.float32)
        action=torch.tensor([b.action for b in batch],dtype=torch.long)
        reward=torch.tensor([b.reward for b in batch],dtype=torch.float32)
        next_obs=torch.tensor(np.stack([b.next_obs for b in batch]),dtype=torch.float32)
        done=torch.tensor([b.done for b in batch],dtype=torch.float32)

        q=online(obs).gather(1,action[:,None]).squeeze(1)
        with torch.no_grad():
            if double_dqn:
                next_action=online(next_obs).argmax(dim=1,keepdim=True)
                next_q=target(next_obs).gather(1,next_action).squeeze(1)
            else:
                next_q=target(next_obs).max(dim=1).values
            y=reward+gamma*(1-done)*next_q

        loss=F.smooth_l1_loss(q,y)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(online.parameters(),10.0)
        optimizer.step()
        return float(loss.item()),float((q.detach()-y).abs().mean().item())
else:
    print("跳过 TD loss 定义的运行测试：需要 PyTorch")

跳过 TD loss 定义的运行测试：需要 PyTorch


In [14]:
if TORCH_AVAILABLE:
    def train_agent(seed,variant="dqn",reward_mode="sparse",cfg=CFG,
                    obs_fn=observation_11,obs_dim=11):
        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
        rng=np.random.default_rng(seed)
        dueling=(variant=="dueling_ddqn")
        double=(variant in {"ddqn","dueling_ddqn"})
        Model=DuelingQNetwork if dueling else QNetwork
        online=Model(obs_dim=obs_dim,hidden=cfg.hidden_size)
        target=Model(obs_dim=obs_dim,hidden=cfg.hidden_size)
        target.load_state_dict(online.state_dict())
        optimizer=optim.Adam(online.parameters(),lr=cfg.learning_rate)
        replay=ReplayBuffer(cfg.replay_capacity)
        env=SnakeEnv(cfg.grid_size,reward_mode,cfg.max_steps_without_food,cfg.gamma)
        env.reset(seed=seed)
        obs=obs_fn(env)
        history=[]

        for step in range(1,cfg.total_env_steps+1):
            eps=epsilon_at(step,cfg)
            if rng.random()<eps:
                action=int(rng.integers(3))
            else:
                with torch.no_grad():
                    x=torch.tensor(obs,dtype=torch.float32).unsqueeze(0)
                    action=int(online(x).argmax(dim=1).item())
            _raw,reward,done,info=env.step(action)
            next_obs=obs_fn(env)
            replay.add(obs,action,reward,next_obs,done)
            obs=next_obs

            if done:
                history.append({"step":step,"train_score":info["score"],
                                "episode_steps":info["steps"],"epsilon":eps})
                env.reset(seed=int(rng.integers(1_000_000_000)))
                obs=obs_fn(env)

            if len(replay)>=cfg.learning_starts and step%cfg.train_every==0:
                batch=replay.sample(cfg.batch_size,rng)
                loss,td=compute_td_loss(batch,online,target,optimizer,cfg.gamma,double)

            if step%cfg.target_update_steps==0:
                target.load_state_dict(online.state_dict())

        return online,history

    def torch_policy(model,obs_fn=observation_11):
        def policy(obs,env,rng):
            with torch.no_grad():
                current_obs=obs_fn(env)
                x=torch.tensor(current_obs,dtype=torch.float32).unsqueeze(0)
                return int(model(x).argmax(dim=1).item())
        return policy
else:
    def train_agent(*args,**kwargs):
        raise RuntimeError("请先安装 PyTorch，再将 TRAIN_MODEL 设为 True")

## 9.1 训练入口与模型保存原则

最终提交时 TRAIN_MODEL=False。训练完成应保存：

- model state_dict；
- 完整 Config；
- variant、reward mode、train seed；
- 代码版本或环境版本；
- 训练历史。

不要只保存整个 Python 对象，因为换版本后较脆弱。模型命名必须含算法、奖励与 seed，防止覆盖。

In [15]:
MODEL_DIR=PROJECT_DIR/"models"
RESULT_DIR=PROJECT_DIR/"results"

if TRAIN_MODEL:
    if not TORCH_AVAILABLE:
        raise RuntimeError("TRAIN_MODEL=True 但 PyTorch 未安装")
    MODEL_DIR.mkdir(exist_ok=True)
    RESULT_DIR.mkdir(exist_ok=True)
    # 正式运行时取消下方注释，并依实验注册表循环多个 seed
    # model,history=train_agent(seed=11,variant="dqn",reward_mode="sparse")
    # torch.save({"state_dict":model.state_dict(),"config":asdict(CFG),
    #             "variant":"dqn","reward_mode":"sparse","seed":11},
    #            MODEL_DIR/"dqn_sparse_seed11.pt")
    print("训练模式已开启")
else:
    print("TRAIN_MODEL=False：不执行长训练。最终提交应从 models/ 载入已保存权重。")

TRAIN_MODEL=False：不执行长训练。最终提交应从 models/ 载入已保存权重。


## E2–E4 必须产出的结果

每个 DQN 变体至少：

1. 5 个训练 seed 的 checkpoint evaluation 曲线；
2. 每条曲线为固定评估局、$\epsilon=0$ 的 game score；
3. seed 均值曲线与不确定性带；
4. 最终 100 局/seed 的 score 分布；
5. 训练耗时和参数量；
6. 典型成功与失败录像；
7. DDQN 额外报告 overestimation gap；
8. Dueling 额外报告 AUC/达到阈值所需 steps。

不要用带探索的 raw training reward 代替正式 evaluation curve；可同时展示，但要清楚标注。

### E2–E4 学习检查

1. Bellman target 为什么在 done 时不含 next-Q？
2. Replay buffer 与 target network 分别解决什么不稳定来源？
3. Vanilla DQN 的 max 为何导致乐观偏差？
4. DDQN 哪个网络选动作，哪个网络估值？
5. Dueling 中为什么要减去平均 advantage？
6. 若 Dueling 参数更多且得分更高，怎样判断提升来自架构还是容量？

# 10. E5：奖励函数消融

固定算法为前面验证后选定的版本（建议 Dueling-DDQN），只改变 R1/R2/R3。研究问题不是“哪个 total reward 大”，而是“哪种反馈让智能体更快、更稳地提高游戏原生 score，且没有投机行为”。

## 理论预测

- Sparse：目标最纯，但早期信用分配困难；
- Step penalty：减少游荡，可能迫使蛇走过于激进的短路；
- Potential shaping：加速方向性学习，理论上较能保留原最优策略，但曼哈顿距离看不见蛇身障碍，仍可能把蛇引入陷阱。

## 现实意义

奖励函数类似企业 KPI。指标设计不当会产生 Goodhart's law：指标成为目标后，便可能不再是好指标。AI 绕圈、拖时间、刷 shaping 奖励，都是“按规则作弊”而非真正变强。

## E5 行为诊断指标

除 score 外记录：

- steps per food：吃一个食物平均花多少步；
- timeout rate：是否绕圈到超时；
- wall/self collision rate：主要死亡方式；
- food approach ratio：多少步缩短了与食物的距离；
- tail-trap case：是否进入局部可行但全局封死的区域。

**假设示例：**

- H5a：R3 在前 50k steps 的学习曲线 AUC 高于 R1；
- H5b：R2 的 steps per food 低于 R1；
- H5c：R3 未必提高最终 score，且可能增加 self collision。

这些是假设，不是预写结论。

In [16]:
EXPERIMENTS = [
    {"id":"P1_size6_greedy","size":6,"policy":"greedy"},
    {"id":"P1_size10_greedy","size":10,"policy":"greedy"},
    {"id":"P1_size15_greedy","size":15,"policy":"greedy"},
    {"id":"E1_pure_random","policy":"pure_random","reward":"sparse"},
    {"id":"E1_safe_random","policy":"safe_random","reward":"sparse"},
    {"id":"E1_greedy","policy":"greedy","reward":"sparse"},
    {"id":"E2_tabular_q","variant":"tabular_q","observation":"O1","reward":"sparse"},
    {"id":"E2_dqn","variant":"dqn","observation":"O1","reward":"sparse"},
    {"id":"E2_ddqn","variant":"ddqn","observation":"O1","reward":"sparse"},
    {"id":"E3_ddqn_rays","variant":"ddqn","observation":"O2","reward":"sparse"},
    {"id":"E4_reward_potential","variant":"ddqn","observation":"best","reward":"potential"},
    {"id":"E5_optional","variant":"dueling_ddqn_or_ppo","status":"only_after_core"},
]
print("实验注册项：",len(EXPERIMENTS))
for e in EXPERIMENTS: print(e)


实验注册项： 12
{'id': 'P1_size6_greedy', 'size': 6, 'policy': 'greedy'}
{'id': 'P1_size10_greedy', 'size': 10, 'policy': 'greedy'}
{'id': 'P1_size15_greedy', 'size': 15, 'policy': 'greedy'}
{'id': 'E1_pure_random', 'policy': 'pure_random', 'reward': 'sparse'}
{'id': 'E1_safe_random', 'policy': 'safe_random', 'reward': 'sparse'}
{'id': 'E1_greedy', 'policy': 'greedy', 'reward': 'sparse'}
{'id': 'E2_tabular_q', 'variant': 'tabular_q', 'observation': 'O1', 'reward': 'sparse'}
{'id': 'E2_dqn', 'variant': 'dqn', 'observation': 'O1', 'reward': 'sparse'}
{'id': 'E2_ddqn', 'variant': 'ddqn', 'observation': 'O1', 'reward': 'sparse'}
{'id': 'E3_ddqn_rays', 'variant': 'ddqn', 'observation': 'O2', 'reward': 'sparse'}
{'id': 'E4_reward_potential', 'variant': 'ddqn', 'observation': 'best', 'reward': 'potential'}
{'id': 'E5_optional', 'variant': 'dueling_ddqn_or_ppo', 'status': 'only_after_core'}


# 11. E6：冻结模型后的最终评估

## 三类随机性要分开

1. train seed：网络初始化、探索、训练环境；
2. validation seed：选择超参数和 checkpoint；
3. test seed：方案冻结后的最终报告。

建议固定 test seeds 为公开列表，并只在最终阶段运行。每个训练 seed 对同一 test seed 集评估，可减少环境难度差异。

**主结论单位：** 每个训练 seed 的 100 局平均 score。先在局内汇总，再跨训练 seed 报均值与离散程度，避免 pseudo-replication。

In [17]:
def summarize_rows(rows):
    scores=np.asarray([r["score"] for r in rows],dtype=float)
    steps=np.asarray([r["steps"] for r in rows],dtype=float)
    deaths={}
    for r in rows: deaths[r["death"]]=deaths.get(r["death"],0)+1
    return {
        "n_episodes":len(rows),
        "score_mean":float(scores.mean()),
        "score_std":float(scores.std(ddof=1)) if len(scores)>1 else 0.0,
        "score_median":float(np.median(scores)),
        "score_q25":float(np.quantile(scores,.25)),
        "score_q75":float(np.quantile(scores,.75)),
        "steps_mean":float(steps.mean()),
        "death_counts":deaths,
    }

def bootstrap_mean_ci(values,n_boot=5000,seed=2026):
    values=np.asarray(values,dtype=float)
    rng=np.random.default_rng(seed)
    sims=np.array([rng.choice(values,size=len(values),replace=True).mean()
                   for _ in range(n_boot)])
    return tuple(np.quantile(sims,[.025,.975]))

rows=evaluate_policy(greedy_policy,range(100,200))
print(summarize_rows(rows))
print("episode-level demo CI:",bootstrap_mean_ci([r["score"] for r in rows],1000))

{'n_episodes': 100, 'score_mean': 18.02, 'score_std': 6.31173350181864, 'score_median': 18.0, 'score_q25': 14.0, 'score_q75': 22.0, 'steps_mean': 142.35, 'death_counts': {'self': 87, 'wall': 13}}
episode-level demo CI: (np.float64(16.77), np.float64(19.21125))


## 正确理解置信区间

上方 episode-level bootstrap 只是展示代码。正式算法比较的关键不确定性来自“重新训练会不会得到不同模型”，所以更严谨做法是：

1. 每个 train seed 在 100 test episodes 上算 mean score；
2. 得到 5 个 seed-level means；
3. 对这 5 个值报告 mean ± std，并 bootstrap CI；
4. 由于种子少，不夸大统计显著性，重视效应大小和一致方向。

若做配对比较，配对单位是相同 train-seed protocol 下的 seed-level summary，不是把全部 episode 当独立样本。

## 最终结果表模板

| Method | Reward | Train seeds | Mean score ± SD | Median | Steps/food | Timeout % | Wall % | Self % | Curve AUC |
|---|---|---:|---:|---:|---:|---:|---:|---:|---:|
| Random | Sparse | N/A | 待填 | 待填 | 待填 | 待填 | 待填 | 待填 | N/A |
| Greedy | Sparse | N/A | 待填 | 待填 | 待填 | 待填 | 待填 | 待填 | N/A |
| DQN | Sparse | 5 | 待填 | 待填 | 待填 | 待填 | 待填 | 待填 | 待填 |
| DDQN | Sparse | 5 | 待填 | 待填 | 待填 | 待填 | 待填 | 待填 | 待填 |
| Dueling-DDQN | Sparse | 5 | 待填 | 待填 | 待填 | 待填 | 待填 | 待填 | 待填 |
| Dueling-DDQN | Step | 5 | 待填 | 待填 | 待填 | 待填 | 待填 | 待填 | 待填 |
| Dueling-DDQN | Potential | 5 | 待填 | 待填 | 待填 | 待填 | 待填 | 待填 | 待填 |

若训练预算吃紧：先保证 DQN/DDQN/Dueling 三者各 3 seeds，再增加到 5；不要用 1 seed 做七种花哨方法。

# 12. E7：训练进步可视化

任务书明确要求 visually demonstrate improvement。建议保存 0%、25%、50%、75%、100% 训练 checkpoint，每个 checkpoint 用相同的 3–5 个 showcase seeds、$\epsilon=0$ 录像。

展示必须同时包含：

- 游戏画面；
- 当前 checkpoint/environment steps；
- score、snake length、当前动作；
- 同一个 seed 的早中晚并排或连续 montage；
- 最终模型完整一局；
- 失败案例，而非只放最高分片段。

学习曲线证明总体趋势，录像解释行为变化；二者不可互相替代。

## 可视化图表清单

1. 游戏规则示意图：蛇头、身体、食物、碰撞；
2. Random/Greedy score 箱线图；
3. DQN/DDQN/Dueling 多 seed evaluation learning curves；
4. reward ablation 学习曲线；
5. 最终 score 箱线图或 violin plot；
6. 死亡类型 stacked bar；
7. 样本效率表（AUC、steps-to-threshold）；
8. 训练进度 montage。

图上写清横纵轴、单位、seed 数、阴影含义。不要截 TensorBoard 而不解释。

In [18]:
# 绘图模板：缺少 matplotlib 时安全跳过
if availability.get("matplotlib",False):
    import matplotlib.pyplot as plt
    demo=np.array([r["score"] for r in rows])
    plt.hist(demo,bins=range(int(demo.max())+2),align="left",rwidth=.8)
    plt.xlabel("Game score = food eaten")
    plt.ylabel("Episodes")
    plt.title("Greedy baseline: demo only")
    plt.show()
else:
    print("matplotlib 未安装：最终环境安装后再生成正式图。")

matplotlib 未安装：最终环境安装后再生成正式图。


# 13. 结果日志与文件结构

建议 proj 1 保持：

    project_notebook.ipynb
    snake_env.py
    agents.py
    train.py
    evaluate.py
    requirements.txt
    models/
    results/
    figures/
    videos/
    README.md

Notebook 是最终 essay；可把稳定实现放 py 文件后导入。results 使用 long format，每行一个 method × reward × train_seed × eval_seed × checkpoint，至少包含 score、steps、return、death、wall_time。

训练日志必须逐 checkpoint 保存，避免训练完成后才发现没有学习曲线。

In [19]:
RESULT_COLUMNS=[
    "method","reward_mode","train_seed","eval_seed","checkpoint_step",
    "score","episode_steps","episode_return","death_reason","wall_time_sec",
    "config_hash","environment_version"
]
print("建议结果字段：")
print("\n".join(RESULT_COLUMNS))

建议结果字段：
method
reward_mode
train_seed
eval_seed
checkpoint_step
score
episode_steps
episode_return
death_reason
wall_time_sec
config_hash
environment_version


# 14. 五人分工与 20 分钟视频（修订）

| 成员 | 核心负责 | 必须交付 | 约 4 分钟讲述 |
|---|---|---|---|
| 1 | 游戏、MDP、环境、棋盘 pilot | 单元测试、尺寸选择证据 | 规则、S/A/P/R/$\gamma$、为何选主棋盘 |
| 2 | 基线与 Tabular Q | Random/Greedy/Q-table | TD update、简单方法上限 |
| 3 | DQN/DDQN | 训练、overestimation | replay、target、Double target |
| 4 | 状态表示与条件扩展 | O1/O2；可选 Dueling/PPO | 信息损失、参数控制、扩展取舍 |
| 5 | 奖励、统计、可视化 | reward ablation、曲线、montage | 科研协议、最终证据与局限 |

Dueling 不再为了“模型多”而强制加入；只有核心实验完成且预算足够才进入。每个人仍须理解全项目。


## 视频故事线（目标 18.5–19.5 分钟）

1. 0:00–4:00：游戏、MDP、棋盘 pilot 与最终尺寸证据；
2. 4:00–8:00：Random/Greedy/Tabular Q，从 Bellman update 进入学习；
3. 8:00–12:00：DQN→DDQN，学习曲线与过估计；
4. 12:00–16:00：O1 vs O2 状态表示；有余力再讲 Dueling/PPO；
5. 16:00–20:00：奖励消融、多 seed 终评、montage、局限。

所有成员真人讲解。不要把时间用于罗列模型名；每个选择都回答“为什么、证据是什么、限制是什么”。


# 15. 最终 Notebook 的 essay 结构

1. Title、成员、贡献声明、AI 使用声明；
2. Abstract：问题、方法、最关键真实结果；
3. Game introduction：规则与可视化；
4. MDP formulation：state/observation/action/reward/termination；
5. Methodology：Random、Greedy、DQN、DDQN、Dueling；
6. Experimental protocol：seeds、预算、validation/test、metrics；
7. E0 环境验证；
8. E1–E5 分实验写 hypothesis→method→result→interpretation；
9. Final evaluation 与 progression visualization；
10. Limitations：部分可观测、算力、种子数、泛化；
11. Conclusion；
12. Reproducibility、AI usage、contributions、references。

不要把本框架原样提交。最终每节必须用你们的数据替换“待填”，删掉教学口吻，形成连贯论证。

## 每个实验统一写作模板

### Research question
精确到可用数据回答的问题。

### Hypothesis
预先说明方向与机制，但允许被否定。

### Controlled variables
哪些保持相同，唯一改变什么。

### Method
训练预算、seeds、网络、评价局数。

### Metrics
一个 primary metric，加必要的机制诊断指标。

### Results
图、表、置信区间；不挑最好结果。

### Interpretation
结果是否支持假设？机制是否合理？替代解释是什么？

### Practical meaning
这个发现对奖励设计、估值偏差、样本效率或现实决策系统有何启示？

### Limitations
什么没有被本实验证明。

# 16. 合规文本草案

## 贡献声明模板

Member A 负责游戏环境、MDP 定义与单元测试；Member B 负责 Random/Greedy 基线及失败案例；Member C 负责 DQN/DDQN 实现与训练；Member D 负责 Dueling 和奖励消融；Member E 负责评估、统计可视化、视频整合。所有成员共同完成实验设计、结果讨论和最终审阅。

必须换成真实姓名并如实调整。

## 生成式 AI 使用声明模板

本项目使用生成式 AI 协助梳理强化学习概念、讨论实验设计、解释报错及润色部分文字。所有环境实现、实验运行、结果核验、图表解释和最终结论均由小组成员审查并负责。我们未使用合成 presenter 或合成声音制作视频。

必须按你们实际用途修改；遗漏使用或虚假声明可能触发作业中的零分规定。

# 17. 里程碑与 Stop/Go 规则（修订）

- T-10 周：环境测试；6/10/15 baseline pilot；冻结主棋盘；
- T-9 周：Tabular Q 多 seed，确保全组理解更新；
- T-8 周：DQN 单 seed 20k–50k smoke test；
- T-7 周：DQN/DDQN 至少 3 seeds；
- T-6 周：实现 O2 30D 并做 observation 单元测试；
- T-5 周：O1/O2 表示实验；
- T-4 周：Sparse/Potential reward ablation；
- T-3 周：若核心完成，再从 Dueling/PPO 中选一个；冻结模型；
- T-2 周：最终 test、Notebook essay、录像；
- T-1 周：真人视频彩排、干净环境运行；
- T-2 天：提前提交。

**Stop rule：** DQN 未稳定就不加 Dueling/PER/PPO；O2 未验证就不做 CNN；多 seed 未完成就不增加模型。


# 18. 风险登记表

| 风险 | 诊断信号 | 应对 |
|---|---|---|
| 环境 bug | 基线异常、不可复现 | 单元测试、人工逐步画面 |
| DQN 发散 | Q 值爆炸、loss 不降 | target、Huber、clip、奖励尺度 |
| 奖励投机 | score 不升但 return 升 | 以 game score 主评、录像诊断 |
| 11维信息不足 | 长蛇阶段稳定失败 | 诚实写 limitation；可选 full-grid |
| 统计不足 | seed 间方向相反 | 增 seed、报告区间、不夸大 |
| 算力不足 | 完整矩阵跑不完 | 先 3 seeds 核心实验、删可选项 |
| 最终 notebook 不能跑 | 缺库/缺模型/路径错误 | 干净环境测试、相对路径、TRAIN_MODEL=False |
| 视频超时 | 彩排 >20 min | 目标 18.5–19.5 min |
| AI 合规遗漏 | 无声明 | 记录实际使用并如实声明 |

# 19. 课程要求对齐检查

| 正式要求 | 本框架落点 |
|---|---|
| 游戏介绍与可视化 | 第 2、12 章 |
| baseline | E1 Random + Greedy |
| learning curves | E2–E5 checkpoint evaluation |
| 多 episodes 评估 | E6 每模型 100 局/seed |
| 展示训练进步 | E7 五个 checkpoint montage |
| 探索提升强度 | DDQN、Dueling、奖励消融 |
| 实验严谨性 | 多 seed、控制变量、validation/test 分离 |
| 可运行 notebook | TRAIN_MODEL=False、保存权重、依赖清单 |
| 5 人视频 20 分钟 | 第 14 章 |
| 贡献声明 | 第 16 章 |
| AI 使用声明 | 第 16 章 |

游戏复杂度与最终强度会计分，因此报告既要解释 Snake 的长期规划难点，也要展示最终模型的真实水平和失败边界。

# 20. 全组开工前必须回答的 12 个问题

1. 我们的动作到底是 3 个相对动作还是 4 个绝对动作？
2. 11 维每一维如何计算？
3. 为什么 11 维不是完整 Markov state？
4. score 与 shaped return 为什么不能混？
5. timeout 如何定义？
6. Random 是 pure random 还是 safe-random？
7. 每个实验唯一改变什么？
8. train/validation/test seeds 如何分？
9. DQN target 和 DDQN target 有何不同？
10. Dueling 为何减 mean advantage？
11. learning curve 是 raw training return 还是固定 checkpoint evaluation？
12. 最终 notebook 在没有重新训练时如何加载模型并运行？

若五人不能共同回答，先不要扩大算法范围。

## 下一步建议（修订后）

第一步跑 E0、Pure/Safe Random、Greedy 和棋盘 pilot；用时间与难度证据决定是否冻结 10×10。第二步训练 Tabular Q，让五人都能解释每一次 Q 更新。第三步只跑一个短 DQN seed，确认学习信号后再启动 DQN/DDQN 多 seed。

推荐最终主线：

$$\text{Baselines}\to\text{Tabular Q}\to\text{DQN}\to\text{DDQN}\to\text{O1 vs O2}\to\text{Reward ablation}.$$

Dueling 或 PPO 只能选一个作扩展，也可以因数据表明“不值得加入”而省略。把棋盘 pilot、Tabular Q 曲线或两个 state-aliasing 局面发给我，我可以继续帮你们诊断并把真实结果写入最终报告。
